# Research Agent

**Build a research agent with LangGraph, GPT-4o, RAG, Pinecone, ArXiv, and Google SerpAPI.**

This notebook is filled in the same order as the source project.

**Corpus**

1. Pull recent `cs.AI` papers from the ArXiv API into a DataFrame and a JSON file.
2. Download each paper PDF.
3. Split every PDF into overlapping chunks and expand the table so each chunk is a row, linked to its neighbors.
4. Embed text with `text-embedding-3-small`.
5. Create the Pinecone index `langgraph-research-agent` and upsert the chunks.

**Tools and the Oracle**

6. Fetch an ArXiv abstract by id.
7. Search the web with Google SerpAPI.
8. Retrieve chunks from Pinecone, with or without an ArXiv-id filter.
9. Compile a research report with `final_answer`.
10. Initialize the Oracle (`gpt-4o`) that chooses those tools, and the scratchpad that records what it already tried.

Cells that need `OPENAI_API_KEY`, `PINECONE_API_KEY`, or `SERPAPI_API_KEY` skip with a short message when the key is missing. The LangGraph loop, the router, and the formatted report are in sections 12–15.


## 01 — Extracting data from ArXiv into a pandas DataFrame and saving it as JSON

In [1]:
# One-time setup. Safe to re-run.
# %pip install -r requirements.txt -q
import json
import os
from pathlib import Path

import pandas as pd
import requests
import xml.etree.ElementTree as ET

# Namespace for ArXiv's Atom-based XML format.
ARXIV_NAMESPACE = "{http://www.w3.org/2005/Atom}"
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent


def extract_from_arxiv(search_query="cat:cs.AI", max_results=100, json_file_path="data/arxiv_dataset.json"):
    """Fetches papers from the ArXiv API based on a search query, saves them as JSON,
    and returns a pandas DataFrame.

    Args:
        search_query (str): The search query for ArXiv (default is 'cat:cs.AI').
        max_results (int): The maximum number of results to retrieve (default is 100).
        json_file_path (str): File path where JSON data will be saved.

    Returns:
        pd.DataFrame: DataFrame containing the extracted paper information.
    """
    url = (
        "http://export.arxiv.org/api/query"
        f"?search_query={search_query}&start=0&max_results={max_results}"
        "&sortBy=submittedDate&sortOrder=descending"
    )
    response = requests.get(
        url,
        headers={"User-Agent": "research-agent-portfolio/1.0 (educational)"},
        timeout=60,
    )
    response.raise_for_status()
    root = ET.fromstring(response.content)

    papers = []
    for entry in root.findall(f"{ARXIV_NAMESPACE}entry"):
        title = " ".join(entry.find(f"{ARXIV_NAMESPACE}title").text.split())
        summary = " ".join(entry.find(f"{ARXIV_NAMESPACE}summary").text.split())
        author_elements = entry.findall(f"{ARXIV_NAMESPACE}author")
        authors = [
            author.find(f"{ARXIV_NAMESPACE}name").text
            for author in author_elements
            if author.find(f"{ARXIV_NAMESPACE}name") is not None
        ]
        paper_url = entry.find(f"{ARXIV_NAMESPACE}id").text.strip()
        arxiv_id = paper_url.split("/")[-1]
        pdf_link = next(
            (
                link.attrib["href"]
                for link in entry.findall(f"{ARXIV_NAMESPACE}link")
                if link.attrib.get("title") == "pdf"
            ),
            None,
        )
        papers.append(
            {
                "title": title,
                "summary": summary,
                "authors": authors,
                "arxiv_id": arxiv_id,
                "url": paper_url,
                "pdf_link": pdf_link,
            }
        )

    out = ROOT / json_file_path
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8") as f:
        json.dump(papers, f, ensure_ascii=False, indent=4)
    print(f"Data saved to {out.relative_to(ROOT)}")
    return pd.DataFrame(papers)


In [2]:
# 20 keeps this notebook quick to re-run. Use 100 for the full project corpus.
df = extract_from_arxiv(max_results=20)
df.head()


Data saved to data/arxiv_dataset.json


,title,summary,authors,arxiv_id,url,pdf_link
0,PlayCraft: Designing Engaging Games from Agent...,"Recent AI agents can generate playable games, ...",[Weihao Cheng],2611.26624v1,http://arxiv.org/abs/2611.26624v1,https://arxiv.org/pdf/2611.26624v1
1,"One Prior, Two Dynamics: Repurposing Video Dif...","Videos reveal how the world changes, but rarel...","[Yuxin Jiang, Wenqiang Sun, Jiaqi Li, Yuchao G...",2611.26617v1,http://arxiv.org/abs/2611.26617v1,https://arxiv.org/pdf/2611.26617v1
2,SoK: Assessing the Evolution of LLM Capability...,LLM-based agents are becoming increasingly cap...,"[Amine Lbath, Manan Suri, Aurelien Delaitre, V...",2611.26601v1,http://arxiv.org/abs/2611.26601v1,https://arxiv.org/pdf/2611.26601v1
3,Active Feature Acquisition by Distilling an Or...,Acquiring every feature of an instance is ofte...,"[Yang Meng, Zixin Ding, Robert L. Grossman, Yu...",2611.26598v1,http://arxiv.org/abs/2611.26598v1,https://arxiv.org/pdf/2611.26598v1
4,PGX: Physically Grounded Execution for Long-Ho...,Robotic vision-language-action (VLA) models ca...,"[Kai Lu, Chenyang Ma, Ioannis Havoutis]",2611.26596v1,http://arxiv.org/abs/2611.26596v1,https://arxiv.org/pdf/2611.26596v1


In [3]:
file_name = ROOT / "data" / "arxiv_dataset.json"
with open(file_name, "r", encoding="utf-8") as file:
    data = json.load(file)
print(f"{len(data)} papers in {file_name.name}")
pd.DataFrame(data).sample(n=min(5, len(data)), random_state=7)[["title", "arxiv_id", "pdf_link"]]


20 papers in arxiv_dataset.json


,title,arxiv_id,pdf_link
1,"One Prior, Two Dynamics: Repurposing Video Dif...",2611.26617v1,https://arxiv.org/pdf/2611.26617v1
17,Backtracking Agent Applications Security Incid...,2611.26525v1,https://arxiv.org/pdf/2611.26525v1
2,SoK: Assessing the Evolution of LLM Capability...,2611.26601v1,https://arxiv.org/pdf/2611.26601v1
5,Realized versus Population Reliability: Anytim...,2611.26592v1,https://arxiv.org/pdf/2611.26592v1
11,MM-SystemOneBias: Evaluating Bias in Multimoda...,2611.26557v1,https://arxiv.org/pdf/2611.26557v1


## 02 — Downloading the research papers (PDFs)

In [4]:
def download_pdfs(df, download_folder="files"):
    """Downloads PDFs from URLs listed in the DataFrame and saves them to a specified folder.
    The file names are stored in a new column 'pdf_file_name' in the DataFrame.

    Args:
        df (pd.DataFrame): DataFrame containing a 'pdf_link' column with URLs to download.
        download_folder (str): Path to the folder where PDFs will be saved (default is 'files').

    Returns:
        pd.DataFrame: The original DataFrame with an additional 'pdf_file_name' column containing
            the paths of the downloaded PDF files or None if the download failed.
    """
    folder = ROOT / download_folder
    if not os.path.exists(folder):
        os.makedirs(folder)

    pdf_file_names = []
    for _, row in df.iterrows():
        pdf_link = row["pdf_link"]
        if not isinstance(pdf_link, str) or not pdf_link.strip():
            print("Skipped a paper with no PDF link")
            pdf_file_names.append(None)
            continue
        try:
            response = requests.get(
                pdf_link,
                headers={"User-Agent": "research-agent-portfolio/1.0 (educational)"},
                timeout=90,
            )
            response.raise_for_status()
            stem = pdf_link.rstrip("/").split("/")[-1]
            if not stem.endswith(".pdf"):
                stem += ".pdf"
            file_name = os.path.join(folder, stem)
            with open(file_name, "wb") as f:
                f.write(response.content)
            rel = os.path.relpath(file_name, ROOT)
            print(f"PDF downloaded successfully and saved as {rel}")
            pdf_file_names.append(rel)
        except requests.exceptions.RequestException as e:
            print(f"Failed to download the PDF: {e}")
            pdf_file_names.append(None)

    out = df.copy()
    out["pdf_file_name"] = pdf_file_names
    return out


In [5]:
# Download the first 3 papers so the chunking section has real PDFs to split.
df = download_pdfs(df.head(3).copy())
df[["title", "arxiv_id", "pdf_file_name"]]


PDF downloaded successfully and saved as files/2611.26624v1.pdf


PDF downloaded successfully and saved as files/2611.26617v1.pdf


PDF downloaded successfully and saved as files/2611.26601v1.pdf


,title,arxiv_id,pdf_file_name
0,PlayCraft: Designing Engaging Games from Agent...,2611.26624v1,files/2611.26624v1.pdf
1,"One Prior, Two Dynamics: Repurposing Video Dif...",2611.26617v1,files/2611.26617v1.pdf
2,SoK: Assessing the Evolution of LLM Capability...,2611.26601v1,files/2611.26601v1.pdf


## 03 — Loading and splitting PDF files into chunks, expanding the DataFrame

In [6]:
import warnings

warnings.filterwarnings("ignore", message=".*langchain-community.*")

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def load_and_chunk_pdf(pdf_file_name, chunk_size=512):
    """Loads a PDF file and splits its content into chunks of a specified size.

    Args:
        pdf_file_name (str): Path to the PDF file to be loaded.
        chunk_size (int): The maximum size of each chunk in characters (default is 512).

    Returns:
        list[Document]: A list of document chunks.
    """
    path = pdf_file_name if os.path.isabs(pdf_file_name) else str(ROOT / pdf_file_name)
    print(f"Loading and splitting into chunks: {os.path.relpath(path, ROOT)}")
    loader = PyPDFLoader(path)
    data = loader.load()
    # Split the content into chunks with slight overlap to preserve context.
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=64)
    chunks = text_splitter.split_documents(data)
    return chunks


def expand_df(df):
    """Expands each row in the DataFrame by splitting PDF documents into chunks.

    Args:
        df (pd.DataFrame): DataFrame containing 'pdf_file_name', 'arxiv_id', 'title', 'summary',
            'authors', and 'url' columns.

    Returns:
        pd.DataFrame: A new DataFrame where each row represents a chunk of the original document,
            with additional metadata such as chunk identifiers and relationships to adjacent chunks.
    """
    expanded_rows = []
    for _, row in df.iterrows():
        if not isinstance(row["pdf_file_name"], str):
            continue
        try:
            chunks = load_and_chunk_pdf(row["pdf_file_name"])
        except Exception as e:
            print(f"Error processing file {row['pdf_file_name']}: {e}")
            continue
        for i, chunk in enumerate(chunks):
            prechunk_id = i - 1 if i > 0 else ""
            postchunk_id = i + 1 if i < len(chunks) - 1 else ""
            expanded_rows.append(
                {
                    "id": f"{row['arxiv_id']}#{i}",
                    "title": row["title"],
                    "summary": row["summary"],
                    "authors": row["authors"],
                    "arxiv_id": row["arxiv_id"],
                    "url": row["url"],
                    "chunk": chunk.page_content,
                    "prechunk_id": "" if i == 0 else f"{row['arxiv_id']}#{prechunk_id}",
                    "postchunk_id": ""
                    if i == len(chunks) - 1
                    else f"{row['arxiv_id']}#{postchunk_id}",
                }
            )
    return pd.DataFrame(expanded_rows)


In [7]:
expanded_df = expand_df(df)
print(f"{len(expanded_df)} chunks from {expanded_df['arxiv_id'].nunique()} papers")
expanded_df[["id", "title", "prechunk_id", "postchunk_id", "chunk"]].head(3)


Loading and splitting into chunks: files/2611.26624v1.pdf


Loading and splitting into chunks: files/2611.26617v1.pdf


Loading and splitting into chunks: files/2611.26601v1.pdf


594 chunks from 3 papers


,id,title,prechunk_id,postchunk_id,chunk
0,2611.26624v1#0,PlayCraft: Designing Engaging Games from Agent...,,2611.26624v1#1,PLAYCRAFT: DESIGNING ENGAGING GAMES\nFROM AGEN...
1,2611.26624v1#1,PlayCraft: Designing Engaging Games from Agent...,2611.26624v1#0,2611.26624v1#2,ence to revise them for mastery. Starting from...
2,2611.26624v1#2,PlayCraft: Designing Engaging Games from Agent...,2611.26624v1#1,2611.26624v1#3,"proficiency levels, and compares those predict..."


## 04 — Building a knowledge base for the RAG system using embeddings

`semantic_router.encoders.OpenAIEncoder` wraps OpenAI's embedding API. The cell below checks for `OPENAI_API_KEY` and does not prompt, so the notebook stays runnable in a fresh clone.


In [8]:
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)

openai_key = os.getenv("OPENAI_API_KEY")
if not openai_key:
    encoder = None
    print("OPENAI_API_KEY is not set. Skipping the embedding encoder until you add it to .env.")
else:
    from semantic_router.encoders import OpenAIEncoder

    encoder = OpenAIEncoder(name="text-embedding-3-small")
    sample = encoder(["hello hallo hola salut"])
    print("Embedding dimensions:", len(sample[0]))


OPENAI_API_KEY is not set. Skipping the embedding encoder until you add it to .env.


## 05 — Creating a Pinecone index

The index is serverless in AWS `us-east-1`, named `langgraph-research-agent`, and sized to the embedding dimension from section 04. Upsert of the chunk corpus continues after this index exists.


In [9]:
import time

from pinecone import Pinecone, ServerlessSpec

index_name = "langgraph-research-agent"
pinecone_key = os.getenv("PINECONE_API_KEY")

if encoder is None or not pinecone_key:
    pc = None
    index = None
    dims = None
    print("Pinecone index was not created. Set OPENAI_API_KEY and PINECONE_API_KEY, then re-run this cell.")
else:
    dims = len(encoder(["hello hallo hola salut"])[0])
    pc = Pinecone(api_key=pinecone_key)
    spec = ServerlessSpec(cloud="aws", region="us-east-1")
    # Check if the index exists; create it if it doesn't.
    if index_name not in pc.list_indexes().names():
        pc.create_index(
            index_name,
            dimension=dims,  # Embedding dimension (1536)
            metric="cosine",
            spec=spec,  # Cloud provider and region specification
        )
        # Wait until the index is fully initialized.
        while not pc.describe_index(index_name).status["ready"]:
            time.sleep(1)
    # Connect to the index.
    index = pc.Index(index_name)
    time.sleep(1)
    print(index.describe_index_stats())


Pinecone index was not created. Set OPENAI_API_KEY and PINECONE_API_KEY, then re-run this cell.


## 06 — Populating the knowledge base and uploading it to Pinecone

Chunks are embedded in batches of 64 and upserted with `arxiv_id`, `title`, and `chunk` metadata.

In [10]:
expanded_df.iloc[:5]

,id,title,summary,authors,arxiv_id,url,chunk,prechunk_id,postchunk_id
0,2611.26624v1#0,PlayCraft: Designing Engaging Games from Agent...,"Recent AI agents can generate playable games, ...",[Weihao Cheng],2611.26624v1,http://arxiv.org/abs/2611.26624v1,PLAYCRAFT: DESIGNING ENGAGING GAMES\nFROM AGEN...,,2611.26624v1#1
1,2611.26624v1#1,PlayCraft: Designing Engaging Games from Agent...,"Recent AI agents can generate playable games, ...",[Weihao Cheng],2611.26624v1,http://arxiv.org/abs/2611.26624v1,ence to revise them for mastery. Starting from...,2611.26624v1#0,2611.26624v1#2
2,2611.26624v1#2,PlayCraft: Designing Engaging Games from Agent...,"Recent AI agents can generate playable games, ...",[Weihao Cheng],2611.26624v1,http://arxiv.org/abs/2611.26624v1,"proficiency levels, and compares those predict...",2611.26624v1#1,2611.26624v1#3
3,2611.26624v1#3,PlayCraft: Designing Engaging Games from Agent...,"Recent AI agents can generate playable games, ...",[Weihao Cheng],2611.26624v1,http://arxiv.org/abs/2611.26624v1,the baselines on the macro-average of each met...,2611.26624v1#2,2611.26624v1#4
4,2611.26624v1#4,PlayCraft: Designing Engaging Games from Agent...,"Recent AI agents can generate playable games, ...",[Weihao Cheng],2611.26624v1,http://arxiv.org/abs/2611.26624v1,A game supports mastery when players initially...,2611.26624v1#3,2611.26624v1#5


In [11]:
from tqdm.auto import tqdm

if encoder is None or index is None:
    print("Skipping the Pinecone upsert until OPENAI_API_KEY and PINECONE_API_KEY are set.")
else:
    data = expanded_df
    batch_size = 64
    for i in tqdm(range(0, len(data), batch_size)):
        i_end = min(len(data), i + batch_size)
        batch = data[i:i_end].to_dict(orient="records")
        metadata = [
            {
                "arxiv_id": r["arxiv_id"],
                "title": r["title"],
                "chunk": r["chunk"],
            }
            for r in batch
        ]
        ids = [r["id"] for r in batch]
        chunks = [r["chunk"] for r in batch]
        embeds = encoder(chunks)
        index.upsert(vectors=zip(ids, embeds, metadata))
    print(index.describe_index_stats())


Skipping the Pinecone upsert until OPENAI_API_KEY and PINECONE_API_KEY are set.


## 07 — Implementing the ArXiv fetch tool

In [12]:
import re

from langchain_core.tools import tool

# Compile a regular expression pattern to find the abstract in the HTML response.
abstract_pattern = re.compile(
    r'<blockquote class="abstract mathjax">\s*<span class="descriptor">Abstract:</span>\s*(.*?)\s*</blockquote>',
    re.DOTALL,
)


@tool("fetch_arxiv")
def fetch_arxiv(arxiv_id: str) -> str:
    """Fetches the abstract from an ArXiv paper given its ArXiv ID.

    Args:
        arxiv_id (str): The ArXiv paper ID.

    Returns:
        str: The extracted abstract text from the ArXiv paper.
    """
    res = requests.get(
        f"https://arxiv.org/abs/{arxiv_id}",
        headers={"User-Agent": "research-agent-portfolio/1.0 (educational)"},
        timeout=30,
    )
    res.raise_for_status()
    re_match = abstract_pattern.search(res.text)
    return re_match.group(1) if re_match else "Abstract not found."


arxiv_id = "1706.03762"
output = fetch_arxiv.invoke(input={"arxiv_id": arxiv_id})
print(output)


The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature. We show that the Transforme

## 08 — Implementing the web search tool with Google SerpAPI

In [13]:
from serpapi import GoogleSearch

serpapi_key = os.getenv("SERPAPI_API_KEY")
serpapi_params = {
    "engine": "google",
    "api_key": serpapi_key or "",
}


@tool("web_search")
def web_search(query: str) -> str:
    """Finds general knowledge information using a Google search.

    Args:
        query (str): The search query string.

    Returns:
        str: A formatted string of the top search results, including title, snippet, and link.
    """
    if not os.getenv("SERPAPI_API_KEY"):
        return "SERPAPI_API_KEY is not set."
    search = GoogleSearch({
        **serpapi_params,
        "q": query,
        "num": 5,
    })
    results = search.get_dict().get("organic_results", [])
    formatted_results = "\n---\n".join(
        ["\n".join([x["title"], x["snippet"], x["link"]]) for x in results]
    )
    return formatted_results if results else "No results found."


if not serpapi_key:
    print("SERPAPI_API_KEY is not set. web_search is defined and will run once the key is in .env.")
else:
    output = web_search.invoke(input={"query": "water on mars"})
    print(output)


SERPAPI_API_KEY is not set. web_search is defined and will run once the key is in .env.


## 09 — Creating RAG tools for retrieval-augmented generation

In [14]:
def format_rag_contexts(matches: list) -> str:
    """Formats the retrieved context matches into a readable string format.

    Args:
        matches (list): A list of matched documents with metadata.

    Returns:
        str: A formatted string of document titles, chunks, and ArXiv IDs.
    """
    formatted_results = []
    for x in matches:
        text = (
            f"Title: {x['metadata']['title']}\n"
            f"Chunk: {x['metadata']['chunk']}\n"
            f"ArXiv ID: {x['metadata']['arxiv_id']}\n"
        )
        formatted_results.append(text)
    return "\n---\n".join(formatted_results)


@tool
def rag_search_filter(query: str, arxiv_id: str) -> str:
    """Finds information from the ArXiv database using a natural language query and a specific ArXiv ID.

    Args:
        query (str): The search query in natural language.
        arxiv_id (str): The ArXiv ID of the specific paper to filter by.

    Returns:
        str: A formatted string of relevant document contexts.
    """
    if encoder is None or index is None:
        return "The Pinecone knowledge base is not connected. Set OPENAI_API_KEY and PINECONE_API_KEY."
    xq = encoder([query])[0]
    xc = index.query(vector=xq, top_k=6, include_metadata=True, filter={"arxiv_id": arxiv_id})
    return format_rag_contexts(xc["matches"])


@tool("rag_search")
def rag_search(query: str) -> str:
    """Finds specialist information on AI using a natural language query.

    Args:
        query (str): The search query in natural language.

    Returns:
        str: A formatted string of relevant document contexts.
    """
    if encoder is None or index is None:
        return "The Pinecone knowledge base is not connected. Set OPENAI_API_KEY and PINECONE_API_KEY."
    xq = encoder([query])[0]
    xc = index.query(vector=xq, top_k=5, include_metadata=True)
    return format_rag_contexts(xc["matches"])


print(format_rag_contexts([
    {
        "metadata": {
            "title": "Attention Is All You Need",
            "chunk": "We propose a new simple network architecture, the Transformer, based solely on attention mechanisms.",
            "arxiv_id": "1706.03762",
        }
    }
]))


Title: Attention Is All You Need
Chunk: We propose a new simple network architecture, the Transformer, based solely on attention mechanisms.
ArXiv ID: 1706.03762



## 10 — Implementing the final answer generation tool

In [15]:
@tool
def final_answer(
    introduction: str,
    research_steps: str | list,
    main_body: str,
    conclusion: str,
    sources: str | list,
) -> str:
    """Returns a natural language response in the form of a research report.

    Args:
        introduction (str): A short paragraph introducing the user's question and the topic.
        research_steps (str or list): Bullet points or text explaining the steps taken for research.
        main_body (str): The bulk of the answer, 3-4 paragraphs long, providing high-quality information.
        conclusion (str): A short paragraph summarizing the findings.
        sources (str or list): A list or text providing the sources referenced during the research.

    Returns:
        str: A formatted research report string.
    """
    if isinstance(research_steps, list):
        research_steps = "\n".join([f"- {r}" for r in research_steps])
    if isinstance(sources, list):
        sources = "\n".join([f"- {s}" for s in sources])
    return (
        f"{introduction}\n\n"
        f"Research Steps:\n{research_steps}\n\n"
        f"Main Body:\n{main_body}\n\n"
        f"Conclusion:\n{conclusion}\n\n"
        f"Sources:\n{sources}"
    )


print(final_answer.invoke({
    "introduction": "This note answers what the Transformer changed in sequence models.",
    "research_steps": ["Fetched the ArXiv abstract for 1706.03762."],
    "main_body": "The paper replaces recurrence with attention, so the model can relate any pair of positions in a sequence in one step.",
    "conclusion": "Attention is sufficient to build a strong sequence model.",
    "sources": ["https://arxiv.org/abs/1706.03762"],
}))


This note answers what the Transformer changed in sequence models.

Research Steps:
- Fetched the ArXiv abstract for 1706.03762.

Main Body:
The paper replaces recurrence with attention, so the model can relate any pair of positions in a sequence in one step.

Conclusion:
Attention is sufficient to build a strong sequence model.

Sources:
- https://arxiv.org/abs/1706.03762


## 11 — Initializing the "Oracle" LLM

The Oracle reads the user question, the chat history, and a scratchpad of tool calls, then must call one of the tools. `gpt-4o` is created only when `OPENAI_API_KEY` is set.

In [16]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_prompt = '''You are the oracle, the great AI decision-maker.
Given the user's query, you must decide what to do with it based on the list of tools provided to you.

If you see that a tool has been used (in the scratchpad) with a particular query, do NOT use that same tool with the same query again. Also, do NOT use any tool more than twice (i.e., if the tool appears in the scratchpad twice, do not use it again).

You should aim to collect information from a diverse range of sources before providing the answer to the user. Once you have collected plenty of information to answer the user's question (stored in the scratchpad), use the final_answer tool.'''

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    ("assistant", "scratchpad: {scratchpad}"),
])


In [17]:
from langchain_core.messages import ToolCall


def create_scratchpad(intermediate_steps: list[ToolCall]) -> str:
    """Build the scratchpad text from intermediate tool calls."""
    research_steps = []
    for _, action in enumerate(intermediate_steps):
        if action.log != "TBD":
            research_steps.append(
                f"Tool: {action.tool}, input: {action.tool_input}\n"
                f"Output: {action.log}"
            )
    return "\n---\n".join(research_steps)


if not os.getenv("OPENAI_API_KEY"):
    llm = None
    oracle = None
    print("OPENAI_API_KEY is not set. The Oracle model is not initialized.")
else:
    from langchain_openai import ChatOpenAI

    llm = ChatOpenAI(
        model="gpt-4o",
        openai_api_key=os.environ["OPENAI_API_KEY"],
        temperature=0,
    )
    tools = [
        rag_search_filter,
        rag_search,
        fetch_arxiv,
        web_search,
        final_answer,
    ]
    oracle = (
        {
            "input": lambda x: x["input"],
            "chat_history": lambda x: x["chat_history"],
            "scratchpad": lambda x: create_scratchpad(intermediate_steps=x["intermediate_steps"]),
        }
        | prompt
        | llm.bind_tools(tools, tool_choice="any")
    )
    print("Oracle is ready:", llm.model_name)


class _Step:
    def __init__(self, tool, tool_input, log):
        self.tool = tool
        self.tool_input = tool_input
        self.log = log


print(create_scratchpad([
    _Step("fetch_arxiv", {"arxiv_id": "1706.03762"}, "Abstract text from the paper."),
    _Step("web_search", {"query": "transformer architecture"}, "TBD"),
]))


OPENAI_API_KEY is not set. The Oracle model is not initialized.
Tool: fetch_arxiv, input: {'arxiv_id': '1706.03762'}
Output: Abstract text from the paper.


## 12 — Testing the Oracle and building the decision pipeline

The live Oracle call needs `OPENAI_API_KEY`. The functions below are the decision pipeline from the project: `run_oracle` asks the model for one tool, `router` reads that tool name, and `run_tool` executes it.

In [18]:
from langchain_core.agents import AgentAction

tools = [
    rag_search_filter,
    rag_search,
    fetch_arxiv,
    web_search,
    final_answer,
]

input_text = "What is the ArXiv paper with the ID 1706.03762 all about?"
inputs = {
    "input": input_text,
    "chat_history": [],
    "intermediate_steps": [],
}

if oracle is None:
    print("OPENAI_API_KEY is not set, so the Oracle was not invoked.")
    print("Sample question:", input_text)
else:
    out = oracle.invoke(inputs)
    print(out)
    print(out.tool_calls[0]["name"])
    print(out.tool_calls[0]["args"])


def run_oracle(state: dict) -> dict:
    """Runs the oracle and processes the output to extract tool information.

    Args:
        state (dict): The current state containing the 'intermediate_steps'.

    Returns:
        dict: A new state with updated 'intermediate_steps' including the tool action.
    """
    print("run_oracle")
    print(f"intermediate_steps: {state['intermediate_steps']}")
    if oracle is None:
        raise RuntimeError("Set OPENAI_API_KEY before running the Oracle.")
    out = oracle.invoke(state)
    tool_name = out.tool_calls[0]["name"]
    tool_args = out.tool_calls[0]["args"]
    action_out = AgentAction(
        tool=tool_name,
        tool_input=tool_args,
        log="TBD",
    )
    return {"intermediate_steps": [action_out]}


def router(state: dict) -> str:
    """Determines the next tool to use based on the current state.

    Args:
        state (dict): The current state containing 'intermediate_steps'.

    Returns:
        str: The name of the tool to use next.
    """
    if isinstance(state["intermediate_steps"], list):
        return state["intermediate_steps"][-1].tool
    print("Router invalid format")
    return "final_answer"


tool_str_to_func = {
    "rag_search_filter": rag_search_filter,
    "rag_search": rag_search,
    "fetch_arxiv": fetch_arxiv,
    "web_search": web_search,
    "final_answer": final_answer,
}


def run_tool(state: dict) -> dict:
    """Executes the appropriate tool based on the current state.

    Args:
        state (dict): The current state containing the 'intermediate_steps'.

    Returns:
        dict: A new state with updated 'intermediate_steps' including the tool's result.
    """
    tool_name = state["intermediate_steps"][-1].tool
    tool_args = state["intermediate_steps"][-1].tool_input
    print(f"{tool_name}.invoke(input={tool_args})")
    out = tool_str_to_func[tool_name].invoke(input=tool_args)
    action_out = AgentAction(
        tool=tool_name,
        tool_input=tool_args,
        log=str(out),
    )
    return {"intermediate_steps": [action_out]}


sample_state = {
    "intermediate_steps": [
        AgentAction(tool="fetch_arxiv", tool_input={"arxiv_id": "1706.03762"}, log="TBD")
    ]
}
print("router ->", router(sample_state))
print(run_tool(sample_state)["intermediate_steps"][-1].log[:240])


OPENAI_API_KEY is not set, so the Oracle was not invoked.
Sample question: What is the ArXiv paper with the ID 1706.03762 all about?
router -> fetch_arxiv
fetch_arxiv.invoke(input={'arxiv_id': '1706.03762'})


The dominant sequence transduction models are based on complex recurrent or convolutional neural networks in an encoder-decoder configuration. The best performing models also connect the encoder and decoder through an attention mechanism. W


## 13 — Defining the agent state

In [19]:
import operator
from typing import Annotated, List, TypedDict

from langchain_core.agents import AgentAction
from langchain_core.messages import BaseMessage


class AgentState(TypedDict):
    """Represents the state of an agent."""

    input: str
    chat_history: List[BaseMessage]
    intermediate_steps: Annotated[List[tuple[AgentAction, str]], operator.add]


## 14 — Defining the graph for decision-making

In [20]:
from langgraph.graph import END, StateGraph

graph = StateGraph(AgentState)
graph.add_node("oracle", run_oracle)
graph.add_node("rag_search_filter", run_tool)
graph.add_node("rag_search", run_tool)
graph.add_node("fetch_arxiv", run_tool)
graph.add_node("web_search", run_tool)
graph.add_node("final_answer", run_tool)
graph.set_entry_point("oracle")
graph.add_conditional_edges(source="oracle", path=router)
for tool_obj in tools:
    if tool_obj.name != "final_answer":
        graph.add_edge(tool_obj.name, "oracle")
graph.add_edge("final_answer", END)
runnable = graph.compile()
print("Graph compiled with nodes:", list(runnable.get_graph().nodes.keys()))


Graph compiled with nodes: ['__start__', 'oracle', 'rag_search_filter', 'rag_search', 'fetch_arxiv', 'web_search', 'final_answer', '__end__']


## 15 — Generating reports: building a formatted final report

In [21]:
def build_report(output: dict) -> str:
    """Builds a formatted report based on the oracle's output.

    Args:
        output (dict): A dictionary containing the various sections of the report.

    Returns:
        str: A formatted string containing the full research report.
    """
    research_steps = output["research_steps"]
    if isinstance(research_steps, list):
        research_steps = "\n".join([f"- {step}" for step in research_steps])
    sources = output["sources"]
    if isinstance(sources, list):
        sources = "\n".join([f"- {source}" for source in sources])
    return f"""
INTRODUCTION
------------
{output["introduction"]}

RESEARCH STEPS
--------------
{research_steps}

REPORT
------
{output["main_body"]}

CONCLUSION
----------
{output["conclusion"]}

SOURCES
-------
{sources}
"""


sample_report = build_report(
    {
        "introduction": "This note answers what the Transformer changed in sequence models.",
        "research_steps": ["Fetched the ArXiv abstract for 1706.03762."],
        "main_body": "The paper replaces recurrence with attention.",
        "conclusion": "Attention is sufficient to build a strong sequence model.",
        "sources": ["https://arxiv.org/abs/1706.03762"],
    }
)
print(sample_report)

if oracle is None:
    print("Skipping runnable.invoke. Set OPENAI_API_KEY, PINECONE_API_KEY, and SERPAPI_API_KEY to run the full graph.")



INTRODUCTION
------------
This note answers what the Transformer changed in sequence models.

RESEARCH STEPS
--------------
- Fetched the ArXiv abstract for 1706.03762.

REPORT
------
The paper replaces recurrence with attention.

CONCLUSION
----------
Attention is sufficient to build a strong sequence model.

SOURCES
-------
- https://arxiv.org/abs/1706.03762

Skipping runnable.invoke. Set OPENAI_API_KEY, PINECONE_API_KEY, and SERPAPI_API_KEY to run the full graph.
